In [1]:
import numpy as np
import pandas as pd
import os
from constants import *
from run import *
from models import *
from sim import *

/Users/juar705/miniconda3/envs/bacterai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class NeuralNetModel(Model):
    def __init__(self, models_path):
        self.models_path = models_path
        self.models = []
        self.is_trained = False
        super().__init__(self, ModelType.NEURAL_NET)

    @classmethod
    def load_trained_models(cls, models_path):
        obj = cls(models_path)

        for filename in os.listdir(models_path):
            if "bag_model" in filename:
                #change model torch.load to weights only is false to be able to read the files.
                model = torch.load(os.path.join(models_path, filename), map_location=torch.device(net.DEVICE), weights_only=False)
                obj.models.append(model)

        obj.is_trained = True
        return obj

    def check_path(self):
        if not os.path.exists(self.models_path):
            os.makedirs(self.models_path)

    def train(self, X_train, y_train, **kwargs):
        self.check_path()
        self.models = net.train_bagged(X_train, y_train, self.models_path, **kwargs)
        self.is_trained = True

    def evaluate(self, X, clip=True):
        if not self.is_trained:
            raise Exception("Neural net model needs to be trained before evaluating.")

        predictions, variances = net.eval_bagged(X, self.models)
        if clip:
            predictions = np.clip(predictions, 0, 1)
        return predictions, variances

In [ ]:
def train_and_predict_nn(
    config_path,
    csv_file_path,
    ingredient_columns,
    growth_column,
    output_csv_path
):
    # Load config
    with open(config_path, 'r') as file:
        config = json.load(file)

    # Load data
    df = pd.read_csv(csv_file_path)
    df = df.dropna(subset=ingredient_columns + [growth_column])
    
    X_train = df[ingredient_columns].to_numpy()
    y_train = df[growth_column].to_numpy()


    n_ingredients = len(ingredient_columns)
    MODEL_TYPE = ModelType(config["model_type"])
    TRANSFER_MODEL_FOLDER = config.get("transfer_model_folder", None)
    N_BAGS = config.get("n_bags", 25)
    EXPT_FOLDER = config.get("experiment_path", ".")
    new_round_folder = os.path.join(EXPT_FOLDER, f"Round1")
    models_folder = os.path.join(new_round_folder, f"nn_models")

    # Load transfer models if needed
    transfer_models = []
    if TRANSFER_MODEL_FOLDER:
        transfer_model = NeuralNetModel.load_trained_models(TRANSFER_MODEL_FOLDER)
        transfer_models = transfer_model.models

    # Train model
    model = NeuralNetModel(models_folder)
    model.train(
        X_train,
        y_train,
        n_ingredients=n_ingredients,
        n_bags=N_BAGS,
        bag_proportion=1.0,
        epochs=50,
        batch_size=20,
        lr=0.001,
        transfer_models=transfer_models,
    )

    # Predict
    test_x = torch.tensor(X_train, dtype=torch.float32)
    print(test_x)
    samples, variances = model.evaluate(test_x)

    # Save to CSV
    output_data = pd.DataFrame({
        'y_pred': samples,
        'y_true': y_train,
        'y_var': variances
    })
    output_data.to_csv(output_csv_path, index=False)
    print(f"Data saved to {output_csv_path}")
    
    xtrain_df = pd.DataFrame(X_train, columns=ingredient_columns)
    xtrain_df['y_pred'] = samples
    xtrain_df.to_csv('/Users/juar705/dashboard/bacterAI_dash/P_putida_AG5577_baseline/Round1/x_train_with_ypredNN.csv', index=False)
    print("X_train with y_pred saved to x_train_with_ypredNN.csv")

In [4]:
ingredient_columns = ['d_glucose', 'sodium_citrate', 'sodium_octanoate', 'sodium_acetate', 'sodium_benzoate', 'd_glucose', 'sodium_chloride', 'potassium_chloride', 'mme_trace_minerals', 'urea', 'ammonium_chloride']
growth_column = 'y'
train_and_predict_nn(config_path='config.json', csv_file_path='/Users/juar705/dashboard/bacterAI_dash/P_putida_AG5577_baseline/Round1/mapped_data_2025-04-15_biotek_final_od_data.csv', ingredient_columns=ingredient_columns, growth_column=growth_column, output_csv_path='/Users/juar705/dashboard/bacterAI_dash/P_putida_AG5577_baseline/Round1/predictions_neuralnet.csv')


Bag 0, p=1.00
	EPOCH  1/50 | Train Loss: 0.3087, Train MSE: 0.3087
	EPOCH  2/50 | Train Loss: 0.2575, Train MSE: 0.2575
	EPOCH  3/50 | Train Loss: 0.0994, Train MSE: 0.0994
	EPOCH  4/50 | Train Loss: 0.1011, Train MSE: 0.1011
	EPOCH  5/50 | Train Loss: 0.0443, Train MSE: 0.0443
	EPOCH  6/50 | Train Loss: 0.0360, Train MSE: 0.0360
	EPOCH  7/50 | Train Loss: 0.0389, Train MSE: 0.0389
	EPOCH  8/50 | Train Loss: 0.0671, Train MSE: 0.0671
	EPOCH  9/50 | Train Loss: 0.0426, Train MSE: 0.0426
	EPOCH 10/50 | Train Loss: 0.0461, Train MSE: 0.0461
	EPOCH 11/50 | Train Loss: 0.0293, Train MSE: 0.0293
	EPOCH 12/50 | Train Loss: 0.0315, Train MSE: 0.0315
	EPOCH 13/50 | Train Loss: 0.0200, Train MSE: 0.0200
	EPOCH 14/50 | Train Loss: 0.0246, Train MSE: 0.0246
	EPOCH 15/50 | Train Loss: 0.0221, Train MSE: 0.0221
	EPOCH 16/50 | Train Loss: 0.0264, Train MSE: 0.0264
	EPOCH 17/50 | Train Loss: 0.0269, Train MSE: 0.0269
	EPOCH 18/50 | Train Loss: 0.0339, Train MSE: 0.0339
	EPOCH 19/50 | Train Loss: 0.01